In [ ]:
STAGE='evaluation'
EVAL_HASH='b3b385dcdf26e99ce96a75a8c837afa87cda13169b1e17058e614607cfd0e3f7'
import csv
import gc
import hashlib
import io
import json
import os
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torchvision.ops import box_convert, box_iou

OUT = Path('/kaggle/working')
MOUNT = Path('/kaggle/input')
DIGEST = 'dfbf35a66d13e0b57447cb91b7f0e9dbc96879240ebe2a3d929f05b6bee851df'
REVISION = 'a2bb814dd30d776dcf7e30523b00659f4f141c71'
torch.set_num_threads(2)

def locate(name):
    return next(MOUNT.rglob(name))

def write_json(name, value):
    (OUT / name).write_text(json.dumps(value, indent=2))

def write_csv(name, rows):
    with (OUT / name).open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

manifest_file = locate('formal_split.csv')
assert hashlib.sha256(manifest_file.read_bytes()).hexdigest() == DIGEST
with manifest_file.open() as f:
    MANIFEST = list(csv.DictReader(f))
PARTS = {s: sorted(int(r['source_id']) for r in MANIFEST if r['split'] == s) for s in ('train','validation','test')}
assert [len(PARTS[s]) for s in PARTS] == [3400,800,2000]
assert len({int(r['source_id']) for r in MANIFEST}) == 6200

def annotations():
    path = locate('instances_train2017.json')
    with path.open() as f:
        data = json.load(f)
    needed = {int(r['source_id']) for r in MANIFEST}
    sizes = {r['id']: (r['width'],r['height']) for r in data['images'] if r['id'] in needed}
    category = next(r['id'] for r in data['categories'] if r['name']=='bicycle')
    ground = defaultdict(list)
    for a in data['annotations']:
        if a['category_id'] == category and a['image_id'] in needed:
            assert not a['iscrowd']
            ground[a['image_id']].append(a['bbox'])
    result = {}
    for i in needed:
        w,h = sizes[i]
        b = torch.tensor(ground[i],dtype=torch.float32).reshape(-1,4)
        result[i] = box_convert(b / torch.tensor([w,h,w,h]),'xywh','xyxy')
    return path.parent.parent, result

def matching(overlap, scores, threshold):
    available = np.ones(overlap.shape[1],dtype=bool)
    kept = np.flatnonzero(scores >= threshold)
    order = kept[np.argsort(-scores[kept],kind='stable')]
    for qi in order:
        options = np.flatnonzero(available & (overlap[qi] >= .5))
        if len(options):
            available[options[np.argmax(overlap[qi,options])]] = False
    tp = int((~available).sum())
    return tp, len(kept)-tp, int(available.sum()), available

def checks():
    assert matching(np.zeros((2,0)),np.array([.9,.8]),.5)[:3] == (0,2,0)
    assert matching(np.array([[.8],[.9]]),np.array([.9,.8]),.5)[:3] == (1,1,0)
    assert matching(np.array([[.8,.6],[.2,.9]]),np.array([.9,.8]),.5)[:3] == (2,0,0)
    assert matching(np.array([[.8]]),np.array([.4]),.5)[:3] == (0,0,1)
    torch.manual_seed(11)
    h = head_new()
    q = torch.randn(4,256)
    assert torch.equal(h(q),torch.zeros_like(q))

def head_new():
    h = nn.Sequential(nn.Linear(256,32),nn.ReLU(),nn.Linear(32,256))
    nn.init.zeros_(h[-1].weight)
    nn.init.zeros_(h[-1].bias)
    return h

def shard_records(stage):
    paths = sorted(MOUNT.rglob(stage+'-shard-*.pt'))
    for p in paths:
        yield from torch.load(p,map_location='cpu',weights_only=True)

def prepare(rec, ground):
    i = rec['source_id']
    iou = box_iou(box_convert(rec['boxes'],'cxcywh','xyxy'),ground[i])
    labels = torch.zeros(len(iou))
    pairs = torch.nonzero(iou>=.5)
    uq, ug = set(),set()
    if len(pairs):
        labels[pairs[:,0]] = -1
        for k in torch.argsort(iou[pairs[:,0],pairs[:,1]],descending=True,stable=True).tolist():
            qi,gi = pairs[k].tolist()
            if qi not in uq and gi not in ug:
                uq.add(qi)
                ug.add(gi)
                labels[qi] = 1
    rec['iou'] = iou.numpy()
    rec['labels'] = labels
    return rec

def predict(rec, head=None):
    if head is None:
        return rec['logits'].sigmoid().numpy()
    with torch.no_grad():
        return (rec['logits'] + head(rec['queries'].float()) @ rec['text']).sigmoid().numpy()

checks()
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

started=time.perf_counter()
lock=json.loads(locate('frozen_training.json').read_text())
assert lock['manifest_sha256']==DIGEST and lock['evaluation_code_sha256']==EVAL_HASH
threshold=lock['threshold']
heads={'untouched':None}
for name,digest in lock['heads'].items():
    p=locate(name)
    assert hashlib.sha256(p.read_bytes()).hexdigest()==digest
    h=head_new(); h.load_state_dict(torch.load(p,weights_only=True)); h.eval()
    heads[name.removesuffix('.pt').removeprefix('head-')]=h
_,ground=annotations()
rows=[]; error_rows=[]; observed=set()
for rec in shard_records('test'):
    i,q=rec['source_id'],rec['quality']
    assert (i,q) not in observed and i in PARTS['test']
    observed.add((i,q))
    rec=prepare(rec,ground)
    condition='original' if q==0 else ('matched' if q==(95 if len(ground[i]) else 90) else 'reversed')
    for name,h in heads.items():
        scores=predict(rec,h)
        tp,fp,fn,missed=matching(rec['iou'],scores,threshold)
        rows.append(dict(model=name,source_id=i,condition=condition,bicycles=len(ground[i]),tp=tp,fp=fp,fn=fn))
        for gi in range(len(ground[i])):
            covered=rec['iou'][:,gi]>=.5
            survives=covered & (scores>=threshold)
            error_rows.append(dict(model=name,source_id=i,annotation_index=gi,condition=condition,
                                   missed=int(missed[gi]),coverage=int(covered.any()),survives=int(survives.any())))
assert observed=={(i,q) for i in PARTS['test'] for q in (95,90,0)}
write_csv('test_per_source.csv',rows)
write_csv('test_per_instance.csv',error_rows)
lookup={(r['model'],r['source_id'],r['condition']):r for r in rows}
positive=[i for i in PARTS['test'] if len(ground[i])]
negative=[i for i in PARTS['test'] if not len(ground[i])]
gt=np.array([len(ground[i]) for i in positive])
miss={}
fa={}
for name in heads:
    for c in ('matched','reversed','original'):
        miss[name,c]=np.array([lookup[name,i,c]['fn'] for i in positive])
        fa[name,c]=np.array([lookup[name,i,c]['fp'] for i in negative])
differences=np.stack([miss[f'A-{s}','reversed']-miss[f'A-{s}','matched']-miss[f'B-{s}','reversed']+miss[f'B-{s}','matched'] for s in (11,23,37)])
seed_delta=100*differences.sum(1)/gt.sum()
rng=np.random.default_rng(5182027)
bootstrap=[]
metrics={}
for group,names in [('untouched',['untouched']),('A',[f'A-{s}' for s in (11,23,37)]),('B',[f'B-{s}' for s in (11,23,37)])]:
    for c in ('matched','reversed','original'):
        metrics[group,c]=dict(miss=np.mean([miss[n,c] for n in names],axis=0),fa=np.mean([fa[n,c] for n in names],axis=0),boot_miss=[],boot_fa=[])
for _ in range(20000):
    ip=rng.integers(len(positive),size=len(positive)); ine=rng.integers(len(negative),size=len(negative))
    denom=gt[ip].sum()
    bootstrap.append(100*differences[:,ip].sum(1).mean()/denom)
    for values in metrics.values():
        values['boot_miss'].append(100*values['miss'][ip].sum()/denom)
        values['boot_fa'].append(values['fa'][ine].mean())
low,high=np.quantile(bootstrap,[.025,.975])
decision='Proceed to mitigation' if low>=3 and (seed_delta>0).all() else ('Hold mitigation project' if high<3 else 'Inconclusive')
summary=[]
for (name,c),v in metrics.items():
    summary.append(dict(model=name,condition=c,miss_rate_percent=100*v['miss'].sum()/gt.sum(),
                        miss_ci95=np.quantile(v['boot_miss'],[.025,.975]).tolist(),
                        false_alarms_per_negative=float(v['fa'].mean()),fa_ci95=np.quantile(v['boot_fa'],[.025,.975]).tolist()))
write_json('absolute_metrics.json',summary)
instance={(r['model'],r['source_id'],r['annotation_index'],r['condition']):r for r in error_rows}
types=defaultdict(int)
for r in error_rows:
    if r['condition']=='reversed' and r['missed'] and not instance[r['model'],r['source_id'],r['annotation_index'],'matched']['missed']:
        kind='no candidate coverage' if not r['coverage'] else ('score below threshold' if not r['survives'] else 'candidate assigned to another bicycle')
        types[r['model'],kind]+=1
errors=[dict(model=n,error_type=t,count=types[n,t]) for n in heads for t in ('no candidate coverage','score below threshold','candidate assigned to another bicycle')]
write_csv('error_types.csv',errors)
report=dict(decision=decision,mean_delta_pp=float(seed_delta.mean()),delta_ci95=[float(low),float(high)],
            delta_pp_by_seed=dict(zip(('11','23','37'),seed_delta.tolist())),threshold=threshold,
            test_sources=2000,test_bicycle_instances=int(gt.sum()),manifest_sha256=DIGEST,evaluation_seconds=time.perf_counter()-started)
write_json('formal_result.json',report)
fig,axes=plt.subplots(1,3,figsize=(12,4),layout='constrained')
colors={'matched':'#246b9e','reversed':'#d27032','original':'#777777'}
for j,(field,interval,ylabel) in enumerate([('miss_rate_percent','miss_ci95','Missed bicycles (%)'),('false_alarms_per_negative','fa_ci95','False alarms per negative image')]):
    for k,c in enumerate(('matched','reversed','original')):
        vals=[next(r for r in summary if r['model']==n and r['condition']==c) for n in ('untouched','A','B')]
        y=np.array([r[field] for r in vals]); ci=np.array([r[interval] for r in vals]).T
        axes[j].errorbar(np.arange(3)+(k-1)*.2,y,yerr=np.maximum(0,np.array([y-ci[0],ci[1]-y])),fmt='o',label=c,color=colors[c],capsize=3)
    axes[j].set_xticks(range(3),['Untouched','A','B']);axes[j].set_ylabel(ylabel)
axes[0].legend(fontsize=8)
axes[2].errorbar([0],[seed_delta.mean()],yerr=[[max(0,seed_delta.mean()-low)],[max(0,high-seed_delta.mean())]],fmt='o',capsize=4)
axes[2].scatter([.15]*3,seed_delta,s=15,label='Seeds')
axes[2].axhline(3,color='#d27032',linestyle='--',label='3-point margin');axes[2].axhline(0,color='grey',linewidth=.6)
axes[2].set_xticks([0],['Extra miss-rate increase']);axes[2].set_ylabel('Percentage points');axes[2].legend(fontsize=8)
fig.savefig(OUT/'formal_results.png',dpi=160)
fig.savefig(OUT/'formal_results.svg')
print(json.dumps(report,indent=2))
